# Emotion-Conditioned Dialogue Generation with BART

This notebook runs the whole project from scratch: it downloads DailyDialog, trains three BART models, and evaluates them on the same test set.

| Model | How the target emotion is given |
|---|---|
| **Baseline** | Not given |
| **EmoToken** | An emotion token such as `[EMOTION-HAPPINESS]` at the start of the decoder output |
| **EmoFusion** | A learned emotion vector added to every encoder input embedding |

**How to run:** in Colab, choose Runtime → Change runtime type → **T4 GPU**, then Runtime → **Run all**.

Everything is saved to one Google Drive folder (`MyDrive/EmoCon` by default). Each model is checkpointed after every epoch. If Colab disconnects, open the notebook again and choose Run all: finished models are skipped, and an unfinished model resumes from its last completed epoch. Generated outputs are cached too.

## 1. Setup

In [1]:
import sys, subprocess
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'rouge-score', 'vaderSentiment', 'tabulate'], check=True)

import io, json, math, os, random, time, zipfile, hashlib, urllib.request
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import BartForConditionalGeneration, BartTokenizer, get_linear_schedule_with_warmup

import nltk
for pkg in ['punkt', 'punkt_tab', 'wordnet']:
    nltk.download(pkg, quiet=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else '(no GPU: training will be very slow)')

Mounted at /content/drive
Device: cuda Tesla T4


## 2. Settings

These are the only values you might want to change. `RUN_DIR` is where everything is saved.

In [2]:
RUN_DIR = Path('/content/drive/MyDrive/NeuralNetworks/EmoCon' if IN_COLAB else 'emocon_run')

CONFIG = dict(
    model_name     = 'facebook/bart-base',
    seed           = 42,
    context_turns  = 3,       # previous turns used as context
    max_input_len  = 128,
    max_target_len = 64,
    epochs         = 3,
    lr             = 3e-5,
    weight_decay   = 0.01,
    warmup_steps   = 200,
    batch_size     = 16,
    grad_accum     = 2,       # effective batch size 32
    max_grad_norm  = 1.0,
    fp16           = True,    # mixed precision on the GPU; much faster on a T4
    emo_emb_dim    = 128,     # EmoFusion emotion embedding size
    gen_batch_size = 64,
    num_beams      = 4,
    max_new_tokens = 64,
    no_repeat_ngram_size = 3,
    bootstrap_resamples  = 1000,
)

DATA_DIR = RUN_DIR / 'data'
CKPT_DIR = RUN_DIR / 'checkpoints'
OUT_DIR  = RUN_DIR / 'outputs'
for d in (DATA_DIR, CKPT_DIR, OUT_DIR):
    d.mkdir(parents=True, exist_ok=True)
(RUN_DIR / 'config.json').write_text(json.dumps(CONFIG, indent=2))
print('Saving everything to', RUN_DIR.resolve())


def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


EMOTIONS = ['neutral', 'anger', 'disgust', 'fear', 'happiness', 'sadness', 'surprise']  # DailyDialog ids 0-6
EMO_TOKENS = {e: f'[EMOTION-{e.upper()}]' for e in EMOTIONS}
VALENCE = {'neutral': 'neutral', 'happiness': 'positive', 'surprise': 'positive',
           'anger': 'negative', 'disgust': 'negative', 'fear': 'negative', 'sadness': 'negative'}

Saving everything to /content/drive/MyDrive/NeuralNetworks/EmoCon


## 3. Data

DailyDialog has 13,118 dialogues with an emotion label for every turn, already split by the authors into train (11,118), validation (1,000) and test (1,000) dialogues. Using these official splits keeps every dialogue entirely in one split, so no test conversation is seen during training.

Each turn after the first becomes one example: **context** = up to 3 previous turns joined with ` [SEP] `, **response** = the turn itself, **emotion** = the response turn's label.

The data is downloaded automatically. If the download fails, download `ijcnlp_dailydialog.zip` yourself from http://yanran.li/dailydialog, put it in `RUN_DIR/data/`, and run this cell again.

In [3]:
OFFICIAL_URL = 'http://yanran.li/files/ijcnlp_dailydialog.zip'
SPLITS = ['train', 'validation', 'test']


def parse_official_zip(zip_bytes):
    """Read the official archive: ijcnlp_dailydialog/{split}.zip -> {split}/dialogues_{split}.txt and
    {split}/dialogues_emotion_{split}.txt. Turns are separated by __eou__."""
    def real_files(z):   # skip macOS metadata such as __MACOSX/._train.zip
        return [n for n in z.namelist() if '__MACOSX' not in n and not n.split('/')[-1].startswith('._')]
    outer = zipfile.ZipFile(io.BytesIO(zip_bytes))
    data = {}
    for split in SPLITS:
        inner_name = next(n for n in real_files(outer) if n.split('/')[-1] == f'{split}.zip')
        inner = zipfile.ZipFile(io.BytesIO(outer.read(inner_name)))
        def read(kind):
            name = next(n for n in real_files(inner) if n.split('/')[-1] == f'dialogues_{kind}{split}.txt')
            return inner.read(name).decode('utf-8').splitlines()
        texts, emotions = read(''), read('emotion_')
        data[split] = [
            ([t.strip() for t in line.split('__eou__')[:-1]], [int(x) for x in emo.split()])
            for line, emo in zip(texts, emotions) if line.strip()
        ]
    return data


def load_from_hub():
    """Fallback: Parquet copies of DailyDialog on the Hugging Face Hub."""
    from huggingface_hub import hf_hub_download
    attempts = [
        ('main',                 'data/{split}-00000-of-00001.parquet'),
        ('refs/convert/parquet', 'default/{split}/0000.parquet'),
        ('refs/pr/9',            'data/{split}-00000-of-00001.parquet'),
    ]
    for revision, pattern in attempts:
        try:
            data = {}
            for split in SPLITS:
                path = hf_hub_download('li2017dailydialog/daily_dialog', pattern.format(split=split),
                                       repo_type='dataset', revision=revision)
                df = pd.read_parquet(path)
                data[split] = [([str(t).strip() for t in d], [int(e) for e in em])
                               for d, em in zip(df['dialog'], df['emotion'])]
            print(f'Loaded from the Hugging Face Hub (revision {revision})')
            return data
        except Exception as err:
            print(f'  Hub revision {revision} failed: {type(err).__name__}')
    raise RuntimeError('Could not load DailyDialog from the Hub either.')


def load_dailydialog():
    local = DATA_DIR / 'ijcnlp_dailydialog.zip'
    if local.exists():
        print('Using', local)
        return parse_official_zip(local.read_bytes())
    try:
        print('Downloading', OFFICIAL_URL)
        with urllib.request.urlopen(OFFICIAL_URL, timeout=60) as r:
            raw = r.read()
        local.write_bytes(raw)
        return parse_official_zip(raw)
    except Exception as err:
        print(f'Official download failed ({type(err).__name__}: {err}). Trying the Hugging Face Hub...')
        return load_from_hub()


def build_pairs(dialogues, split):
    pairs, skipped = [], 0
    for d_idx, (turns, emos) in enumerate(dialogues):
        turns = [t for t in turns if t]
        if len(turns) < 2 or len(turns) != len(emos):
            skipped += 1
            continue
        for i in range(1, len(turns)):
            pairs.append({
                'id': f'{split}-{d_idx}-{i}',
                'context': ' [SEP] '.join(turns[max(0, i - CONFIG['context_turns']):i]),
                'response': turns[i],
                'emotion': emos[i],
                'emotion_label': EMOTIONS[emos[i]],
            })
    return pairs, skipped


def fingerprint(pairs):
    h = hashlib.sha256()
    for p in pairs:
        h.update((p['context'] + '\x00' + p['response'] + '\x01').encode())
    return h.hexdigest()[:12]


PAIRS_PATH = DATA_DIR / 'pairs.json'
if PAIRS_PATH.exists():
    pairs = json.loads(PAIRS_PATH.read_text())
    print('Loaded saved pairs from', PAIRS_PATH)
else:
    raw = load_dailydialog()
    pairs, info = {}, {}
    for split in SPLITS:
        pairs[split], skipped = build_pairs(raw[split], split)
        info[split] = {'dialogues': len(raw[split]), 'skipped': skipped, 'pairs': len(pairs[split])}
    pairs['fingerprints'] = {s: fingerprint(pairs[s]) for s in SPLITS}
    PAIRS_PATH.write_text(json.dumps(pairs, ensure_ascii=False))
    print(pd.DataFrame(info).T)

train_pairs, val_pairs, test_pairs = pairs['train'], pairs['validation'], pairs['test']
for s in SPLITS:
    assert fingerprint(pairs[s]) == pairs['fingerprints'][s], f'{s} pairs file is corrupted'

label_table = pd.DataFrame({s: Counter(p['emotion_label'] for p in pairs[s]) for s in SPLITS}).reindex(EMOTIONS).fillna(0).astype(int)
label_table['train %'] = (label_table['train'] / label_table['train'].sum() * 100).round(1)
print({s: len(pairs[s]) for s in SPLITS})
label_table

Official download failed (BadZipFile: File is not a zip file). Trying the Hugging Face Hub...


  Hub revision main failed: RemoteEntryNotFoundError


default/train/0000.parquet: reconstructing file:   0%|          |  0.00B / 3.61MB            

default/train/0000.parquet: downloading bytes:           |  0.00B            

0000.parquet:   0%|          | 0.00/334k [00:00<?, ?B/s]

0000.parquet:   0%|          | 0.00/331k [00:00<?, ?B/s]

Loaded from the Hugging Face Hub (revision refs/convert/parquet)
            dialogues  skipped  pairs
train           11118        0  76052
validation       1000        0   7069
test             1000        0   6740
{'train': 76052, 'validation': 7069, 'test': 6740}


,train,validation,test,train %
neutral,62357,6186,5454,82.0
anger,691,62,102,0.9
disgust,247,3,41,0.3
fear,123,8,14,0.2
happiness,10253,649,927,13.5
sadness,877,65,94,1.2
surprise,1504,96,108,2.0


## 4. Models and training code

**Class imbalance.** Most responses are neutral, so the emotion models weight rarer emotions more. The weight for emotion *c* is the square root of its inverse frequency, scaled so the average weight over training examples is 1. The square root keeps neutral examples from being almost ignored; with plain inverse frequency, a neutral example would count about 50 times less than a rare one.

- **EmoToken:** the weight applies to the loss on the emotion token.
- **EmoFusion:** each example's loss is multiplied by its own weight (per example, not per batch).
- **Baseline:** no weighting.

Validation loss is always unweighted, so the three models' validation losses are comparable.

In [4]:
def emotion_weights(train_pairs):
    counts = Counter(p['emotion'] for p in train_pairs)
    n = len(train_pairs)
    raw = {c: math.sqrt(n / (len(EMOTIONS) * counts.get(c, 1))) for c in range(len(EMOTIONS))}
    mean_over_examples = sum(raw[p['emotion']] for p in train_pairs) / n
    return {c: w / mean_over_examples for c, w in raw.items()}


class PairDataset(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs
    def __len__(self):
        return len(self.pairs)
    def __getitem__(self, i):
        return self.pairs[i]


def make_collate(tokenizer, emotion_tokens=None):
    """Tokenize a batch with dynamic padding. With emotion_tokens, each target becomes
    '<s> [EMOTION-X] response </s>'."""
    def collate(batch):
        enc = tokenizer([p['context'] for p in batch], max_length=CONFIG['max_input_len'],
                        padding=True, truncation=True, return_tensors='pt')
        targets = [p['response'] for p in batch]
        if emotion_tokens is not None:
            targets = [f"{emotion_tokens[p['emotion_label']]} {t}" for p, t in zip(batch, targets)]
        dec = tokenizer(targets, max_length=CONFIG['max_target_len'], padding=True,
                        truncation=True, return_tensors='pt')
        labels = dec['input_ids'].clone()
        labels[dec['attention_mask'] == 0] = -100
        return {'input_ids': enc['input_ids'], 'attention_mask': enc['attention_mask'], 'labels': labels,
                'emotion_ids': torch.tensor([p['emotion'] for p in batch], dtype=torch.long)}
    return collate


class EmotionAwareBart(nn.Module):
    """EmoFusion: BART plus a learned emotion vector added to every encoder input embedding."""
    def __init__(self, bart, n_emotions=len(EMOTIONS), emo_emb_dim=128):
        super().__init__()
        self.bart = bart
        self.emo_embed = nn.Embedding(n_emotions, emo_emb_dim)
        self.emo_proj = nn.Linear(emo_emb_dim, bart.config.d_model)
        nn.init.normal_(self.emo_embed.weight, std=0.02)
        nn.init.normal_(self.emo_proj.weight, std=0.02)
        nn.init.zeros_(self.emo_proj.bias)

    def fused_embeds(self, input_ids, emotion_ids):
        # bart-base does not scale embeddings (scale_embedding=False), so this matches what the
        # encoder would compute from input_ids, plus the emotion vector.
        return self.bart.model.shared(input_ids) + self.emo_proj(self.emo_embed(emotion_ids)).unsqueeze(1)

    def forward(self, input_ids, attention_mask, emotion_ids, labels=None):
        return self.bart(inputs_embeds=self.fused_embeds(input_ids, emotion_ids),
                         attention_mask=attention_mask, labels=labels)

    @torch.no_grad()
    def generate(self, input_ids, attention_mask, emotion_ids, **kw):
        return self.bart.generate(inputs_embeds=self.fused_embeds(input_ids, emotion_ids),
                                  attention_mask=attention_mask, **kw)


# ---- Loss functions: each returns (loss to backpropagate, unweighted loss to log) ----

def token_ce(logits, labels):
    return F.cross_entropy(logits.float().transpose(1, 2), labels, ignore_index=-100, reduction='none')


def baseline_loss(model, b):
    loss = model(input_ids=b['input_ids'], attention_mask=b['attention_mask'], labels=b['labels']).loss
    return loss, loss


def emotoken_loss(weights):
    w_table = torch.tensor([weights[c] for c in range(len(EMOTIONS))])
    def fn(model, b):
        out = model(input_ids=b['input_ids'], attention_mask=b['attention_mask'], labels=b['labels'])
        ce = token_ce(out.logits, b['labels'])
        mask = (b['labels'] != -100).float()
        tw = torch.ones_like(ce)
        tw[:, 1] = w_table.to(ce.device)[b['emotion_ids']]   # position 1 holds the emotion token
        weighted = (ce * tw * mask).sum() / (tw * mask).sum()
        return weighted, out.loss
    return fn


def emofusion_loss(weights):
    w_table = torch.tensor([weights[c] for c in range(len(EMOTIONS))])
    def fn(model, b):
        out = model(b['input_ids'], b['attention_mask'], b['emotion_ids'], labels=b['labels'])
        ce = token_ce(out.logits, b['labels'])
        n_tok = (b['labels'] != -100).sum(1).clamp(min=1)
        per_example = ce.sum(1) / n_tok
        return (per_example * w_table.to(ce.device)[b['emotion_ids']]).mean(), out.loss
    return fn


def train_model(name, model, tokenizer, collate, loss_fn, save_best):
    """Train with resume support. Saves the best epoch (by validation loss) and a resume file after
    every epoch. Skips entirely if this model has already finished."""
    out_dir = CKPT_DIR / name
    out_dir.mkdir(parents=True, exist_ok=True)
    done, resume = out_dir / 'done.json', out_dir / 'resume.pt'
    if done.exists():
        print(f'{name}: already trained, skipping.', json.loads(done.read_text())['history'])
        return

    set_seed(CONFIG['seed'])
    model.to(DEVICE)
    use_amp = CONFIG['fp16'] and DEVICE.type == 'cuda'
    train_loader = DataLoader(PairDataset(train_pairs), batch_size=CONFIG['batch_size'], shuffle=True,
                              collate_fn=collate, num_workers=2, generator=torch.Generator())
    val_loader = DataLoader(PairDataset(val_pairs), batch_size=CONFIG['batch_size'] * 2,
                            collate_fn=collate, num_workers=2)
    optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG['lr'], weight_decay=CONFIG['weight_decay'])
    steps = math.ceil(len(train_loader) / CONFIG['grad_accum']) * CONFIG['epochs']
    scheduler = get_linear_schedule_with_warmup(optimizer, CONFIG['warmup_steps'], steps)
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp)

    start_epoch, best, history = 1, float('inf'), []
    if resume.exists():
        state = torch.load(resume, map_location='cpu', weights_only=False)
        model.load_state_dict(state['model'])
        optimizer.load_state_dict(state['optimizer'])
        scheduler.load_state_dict(state['scheduler'])
        scaler.load_state_dict(state['scaler'])
        start_epoch, best, history = state['epoch'] + 1, state['best'], state['history']
        print(f'{name}: resuming from epoch {start_epoch}')

    def to_device(b):
        return {k: v.to(DEVICE, non_blocking=True) for k, v in b.items()}

    for epoch in range(start_epoch, CONFIG['epochs'] + 1):
        t0 = time.time()
        train_loader.generator.manual_seed(CONFIG['seed'] + epoch)   # same shuffle order after a resume
        model.train()
        optimizer.zero_grad(set_to_none=True)
        running = 0.0
        for step, b in enumerate(tqdm(train_loader, desc=f'{name} epoch {epoch}/{CONFIG["epochs"]}')):
            b = to_device(b)
            with torch.autocast('cuda', dtype=torch.float16, enabled=use_amp):
                loss, logged = loss_fn(model, b)
            scaler.scale(loss / CONFIG['grad_accum']).backward()
            running += logged.item()
            if (step + 1) % CONFIG['grad_accum'] == 0 or step + 1 == len(train_loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG['max_grad_norm'])
                scaler.step(optimizer); scaler.update(); scheduler.step()
                optimizer.zero_grad(set_to_none=True)

        model.eval()
        val = 0.0
        with torch.no_grad():
            for b in tqdm(val_loader, desc='validation', leave=False):
                b = to_device(b)
                with torch.autocast('cuda', dtype=torch.float16, enabled=use_amp):
                    val += loss_fn(model, b)[1].item()
        val /= len(val_loader)
        history.append({'epoch': epoch, 'train_loss': round(running / len(train_loader), 4),
                        'val_loss': round(val, 4), 'minutes': round((time.time() - t0) / 60, 1)})
        print(history[-1])
        if val < best:
            best = val
            save_best(model, tokenizer, out_dir)
            print(f'  saved best model (val loss {val:.4f})')
        torch.save({'model': model.state_dict(), 'optimizer': optimizer.state_dict(),
                    'scheduler': scheduler.state_dict(), 'scaler': scaler.state_dict(),
                    'epoch': epoch, 'best': best, 'history': history}, resume)

    done.write_text(json.dumps({'config': CONFIG, 'history': history, 'best_val_loss': best}, indent=2))
    resume.unlink(missing_ok=True)   # frees about 1.7 GB of Drive space
    print(f'{name}: finished.')


def save_hf(model, tokenizer, out_dir):
    model.save_pretrained(out_dir); tokenizer.save_pretrained(out_dir)


def save_emofusion(model, tokenizer, out_dir):
    model.bart.save_pretrained(out_dir); tokenizer.save_pretrained(out_dir)
    torch.save({'emo_embed': model.emo_embed.state_dict(), 'emo_proj': model.emo_proj.state_dict()},
               out_dir / 'emotion_layers.pt')


WEIGHTS = emotion_weights(train_pairs)
print('Class weights:', {EMOTIONS[c]: round(w, 3) for c, w in WEIGHTS.items()})

Class weights: {'neutral': 0.645, 'anger': 6.124, 'disgust': 10.242, 'fear': 14.514, 'happiness': 1.59, 'sadness': 5.436, 'surprise': 4.151}


## 5. Train the baseline

In [5]:
if not (CKPT_DIR / 'baseline' / 'done.json').exists():
    set_seed(CONFIG['seed'])
    tok = BartTokenizer.from_pretrained(CONFIG['model_name'])
    model = BartForConditionalGeneration.from_pretrained(CONFIG['model_name'])
    train_model('baseline', model, tok, make_collate(tok), baseline_loss, save_hf)
    del model; torch.cuda.empty_cache()
else:
    train_model('baseline', None, None, None, None, None)

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.72k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  558MB            

model.safetensors: downloading bytes:           |  0.00B            

... (13 outputs left out)

validation:   0%|          | 0/221 [00:00<?, ?it/s]

{'epoch': 3, 'train_loss': 2.3488, 'val_loss': 2.3174, 'minutes': 10.7}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  saved best model (val loss 2.3174)
baseline: finished.


## 6. Train EmoToken

Seven emotion tokens are added to the vocabulary. In training, the decoder sees `</s> <s> [EMOTION-X] response…` (BART shifts the target right). Generation later starts from exactly that prompt.

In [6]:
if not (CKPT_DIR / 'emotoken' / 'done.json').exists():
    set_seed(CONFIG['seed'])
    tok = BartTokenizer.from_pretrained(CONFIG['model_name'])
    tok.add_special_tokens({'additional_special_tokens': list(EMO_TOKENS.values())})
    model = BartForConditionalGeneration.from_pretrained(CONFIG['model_name'])
    model.resize_token_embeddings(len(tok))

    # Check the target layout the loss and generation rely on: <s> [EMOTION-X] ...
    sample = make_collate(tok, EMO_TOKENS)([train_pairs[0]])
    assert sample['labels'][0, 0].item() == tok.bos_token_id
    assert sample['labels'][0, 1].item() == tok.convert_tokens_to_ids(EMO_TOKENS[train_pairs[0]['emotion_label']])

    train_model('emotoken', model, tok, make_collate(tok, EMO_TOKENS), emotoken_loss(WEIGHTS), save_hf)
    del model; torch.cuda.empty_cache()
else:
    train_model('emotoken', None, None, None, None, None)

Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

[transformers] The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


emotoken epoch 1/3:   0%|          | 0/4754 [00:00<?, ?it/s]

validation:   0%|          | 0/221 [00:00<?, ?it/s]

{'epoch': 1, 'train_loss': 2.7294, 'val_loss': 2.2942, 'minutes': 11.8}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (7 outputs left out)

validation:   0%|          | 0/221 [00:00<?, ?it/s]

{'epoch': 3, 'train_loss': 2.2518, 'val_loss': 2.2118, 'minutes': 11.8}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  saved best model (val loss 2.2118)
emotoken: finished.


## 7. Train EmoFusion

In [7]:
if not (CKPT_DIR / 'emofusion' / 'done.json').exists():
    set_seed(CONFIG['seed'])
    tok = BartTokenizer.from_pretrained(CONFIG['model_name'])
    model = EmotionAwareBart(BartForConditionalGeneration.from_pretrained(CONFIG['model_name']),
                             emo_emb_dim=CONFIG['emo_emb_dim'])
    train_model('emofusion', model, tok, make_collate(tok), emofusion_loss(WEIGHTS), save_emofusion)
    del model; torch.cuda.empty_cache()
else:
    train_model('emofusion', None, None, None, None, None)

Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

emofusion epoch 1/3:   0%|          | 0/4754 [00:00<?, ?it/s]

/tmp/ipykernel_436/1073181366.py:147: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.step(optimizer); scaler.update(); scheduler.step()


validation:   0%|          | 0/221 [00:00<?, ?it/s]

{'epoch': 1, 'train_loss': 2.8335, 'val_loss': 2.4729, 'minutes': 12.2}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (7 outputs left out)

validation:   0%|          | 0/221 [00:00<?, ?it/s]

{'epoch': 3, 'train_loss': 2.4639, 'val_loss': 2.3936, 'minutes': 12.5}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  saved best model (val loss 2.3936)
emofusion: finished.


## 8. Generate responses for the test set

Every model generates a response for every test example, using beam search. The emotion models are given the reference response's emotion (the oracle setting): this tests whether a model can follow an emotion it is given, not whether it can choose one.

`EmoToken (no <s> in prompt)` is an extra check: it generates from `</s> [EMOTION-X]`, skipping the `<s>` the model saw in training. Comparing it with EmoToken shows how much matching the training layout matters.

Outputs are saved to `RUN_DIR/outputs/` and reused if this cell runs again.

In [8]:
GEN_KW = dict(num_beams=CONFIG['num_beams'], max_new_tokens=CONFIG['max_new_tokens'],
              no_repeat_ngram_size=CONFIG['no_repeat_ngram_size'], early_stopping=True)
contexts   = [p['context'] for p in test_pairs]
references = [p['response'] for p in test_pairs]
targets    = [p['emotion_label'] for p in test_pairs]


def batches(n, size):
    for i in range(0, n, size):
        yield slice(i, i + size)


@torch.no_grad()
def generate_all(model, tok, mode):
    model.to(DEVICE).eval()
    out = []
    for sl in tqdm(list(batches(len(test_pairs), CONFIG['gen_batch_size'])), desc=mode):
        batch = test_pairs[sl]
        enc = tok([p['context'] for p in batch], max_length=CONFIG['max_input_len'], padding=True,
                  truncation=True, return_tensors='pt').to(DEVICE)
        if mode == 'baseline':
            ids = model.generate(**enc, **GEN_KW)
        elif mode in ('emotoken', 'emotoken_no_bos'):
            emo = [tok.convert_tokens_to_ids(EMO_TOKENS[p['emotion_label']]) for p in batch]
            start = model.config.decoder_start_token_id
            prompt = [[start, tok.bos_token_id, e] if mode == 'emotoken' else [start, e] for e in emo]
            ids = model.generate(**enc, decoder_input_ids=torch.tensor(prompt, device=DEVICE),
                                 forced_bos_token_id=None, **GEN_KW)
        elif mode == 'emofusion':
            emo = torch.tensor([p['emotion'] for p in batch], device=DEVICE)
            ids = model.generate(enc['input_ids'], enc['attention_mask'], emo, **GEN_KW)
        out.extend(s.strip() for s in tok.batch_decode(ids, skip_special_tokens=True))
    return out


def load_model(name):
    d = CKPT_DIR / name
    tok = BartTokenizer.from_pretrained(d)
    bart = BartForConditionalGeneration.from_pretrained(d)
    if name != 'emofusion':
        return bart, tok
    model = EmotionAwareBart(bart, emo_emb_dim=CONFIG['emo_emb_dim'])
    layers = torch.load(d / 'emotion_layers.pt', map_location='cpu')
    model.emo_embed.load_state_dict(layers['emo_embed']); model.emo_proj.load_state_dict(layers['emo_proj'])
    return model, tok


RUNS = {  # row name: (checkpoint, generation mode, output file)
    'Baseline':                 ('baseline',  'baseline',        'baseline.csv'),
    'EmoToken':                 ('emotoken',  'emotoken',        'emotoken.csv'),
    'EmoToken (no <s> in prompt)': ('emotoken', 'emotoken_no_bos', 'emotoken_no_bos.csv'),
    'EmoFusion':                ('emofusion', 'emofusion',       'emofusion.csv'),
}
generated = {}
for row, (ckpt, mode, fname) in RUNS.items():
    path = OUT_DIR / fname
    if path.exists():
        df = pd.read_csv(path, keep_default_na=False)
        assert df['id'].tolist() == [p['id'] for p in test_pairs], f'{path} does not match the test set; delete it'
        generated[row] = df['generated'].tolist()
        print(f'{row}: loaded {len(df)} saved outputs')
        continue
    model, tok = load_model(ckpt)
    generated[row] = generate_all(model, tok, mode)
    pd.DataFrame({'id': [p['id'] for p in test_pairs], 'generated': generated[row]}).to_csv(path, index=False)
    del model; torch.cuda.empty_cache()

Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]

baseline:   0%|          | 0/106 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]

emotoken:   0%|          | 0/106 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]

emotoken_no_bos:   0%|          | 0/106 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]

emofusion:   0%|          | 0/106 [00:00<?, ?it/s]

## 9. Evaluate

- **Text quality:** BLEU-4, ROUGE-L and METEOR against the reference response; Distinct-1/2 for variety.
- **Emotion alignment:** an emotion classifier (`j-hartmann/emotion-english-distilroberta-base`) labels each generated response, and the label is compared with the target emotion. Reported for all 7 emotions and grouped into positive / negative / neutral. **Macro-F1** is the main number, because accuracy is dominated by the neutral class.
- **Sentiment coherence:** whether the response has the same VADER sentiment as the last context turn.
- **Reference rows:** *Always neutral* is what a model scores if every response is labelled neutral. *Human reference* is the real responses scored by the same classifier, showing how often the classifier agrees with DailyDialog's labels.

In [9]:
from nltk.tokenize import word_tokenize
from nltk.translate.bleu_score import SmoothingFunction, corpus_bleu
from nltk.translate.meteor_score import meteor_score
from rouge_score import rouge_scorer
from sklearn.metrics import accuracy_score, f1_score
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from transformers import pipeline


def toks(s):
    return word_tokenize(str(s).lower())


def text_quality(refs, hyps):
    sc = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
    def distinct(n):
        grams = [tuple(t[i:i + n]) for h in hyps for t in [toks(h)] for i in range(len(t) - n + 1)]
        return len(set(grams)) / len(grams) * 100 if grams else 0.0
    return {
        'BLEU-4': corpus_bleu([[toks(r)] for r in refs], [toks(h) for h in hyps],
                              smoothing_function=SmoothingFunction().method1) * 100,
        'ROUGE-L': np.mean([sc.score(str(r), str(h))['rougeL'].fmeasure for r, h in zip(refs, hyps)]) * 100,
        'METEOR': np.mean([meteor_score([toks(r)], toks(h)) for r, h in zip(refs, hyps)]) * 100,
        'Distinct-1': distinct(1),
        'Distinct-2': distinct(2),
    }


CLF_MAP = {'anger': 'anger', 'disgust': 'disgust', 'fear': 'fear', 'joy': 'happiness',
           'neutral': 'neutral', 'sadness': 'sadness', 'surprise': 'surprise'}
classifier = pipeline('text-classification', model='j-hartmann/emotion-english-distilroberta-base',
                      top_k=1, device=0 if DEVICE.type == 'cuda' else -1)


def classify(texts, bs=64):
    labels = []
    for sl in batches(len(texts), bs):
        for pred in classifier([str(t)[:512] or '.' for t in texts[sl]], truncation=True):
            top = pred[0] if isinstance(pred, list) else pred
            labels.append(CLF_MAP.get(top['label'].lower(), 'neutral'))
    return labels


def alignment(tgt, pred):
    t3, p3 = [VALENCE[x] for x in tgt], [VALENCE[x] for x in pred]
    return {
        'Emotion macro-F1 (7)': f1_score(tgt, pred, average='macro', zero_division=0) * 100,
        'Emotion acc (7)': accuracy_score(tgt, pred) * 100,
        'Emotion macro-F1 (3)': f1_score(t3, p3, average='macro', zero_division=0) * 100,
        'Emotion acc (3)': accuracy_score(t3, p3) * 100,
    }


vader = SentimentIntensityAnalyzer()
def polarity(t):
    s = vader.polarity_scores(str(t))['compound']
    return 'positive' if s >= 0.05 else 'negative' if s <= -0.05 else 'neutral'
last_turn_polarity = [polarity(c.split('[SEP]')[-1]) for c in contexts]


predicted = {row: classify(gen) for row, gen in generated.items()}
predicted['Human reference'] = classify(references)
predicted['Always neutral'] = ['neutral'] * len(test_pairs)

rows = []
for row, preds in predicted.items():
    r = {'Model': row}
    if row in generated:
        r.update(text_quality(references, generated[row]))
        r['Sentiment coherence'] = np.mean([polarity(h) == c for h, c in zip(generated[row], last_turn_polarity)]) * 100
    r.update(alignment(targets, preds))
    rows.append(r)
results = pd.DataFrame(rows).set_index('Model').round(2)
results

config.json:   0%|          | 0.00/1.00k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  329MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/294 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  329MB            

... (3 outputs left out)

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


,BLEU-4,ROUGE-L,METEOR,Distinct-1,Distinct-2,Sentiment coherence,Emotion macro-F1 (7),Emotion acc (7),Emotion macro-F1 (3),Emotion acc (3)
Model,,,,,,,,,,
Baseline,2.73,15.89,15.39,4.08,18.66,53.12,18.60,54.69,38.66,56.34
EmoToken,3.05,16.57,16.20,4.09,19.11,51.74,27.95,63.18,47.54,64.38
EmoToken (no <s> in prompt),2.77,15.85,15.37,4.10,18.89,53.01,19.23,52.92,38.90,54.64
EmoFusion,1.85,15.24,14.21,4.50,19.42,50.74,28.44,65.16,50.17,66.69
Human reference,NaN,NaN,NaN,NaN,NaN,NaN,28.38,63.74,49.02,65.91
Always neutral,NaN,NaN,NaN,NaN,NaN,NaN,12.78,80.92,29.82,80.92


### Are the differences bigger than chance?

A paired bootstrap resamples the test examples 1,000 times and recomputes the difference in macro-F1 (7 emotions) each time. If the 95% interval includes 0, the difference is not reliable.

In [10]:
def paired_bootstrap(tgt, a, b, n=CONFIG['bootstrap_resamples'], seed=0):
    t, a, b = np.array(tgt), np.array(a), np.array(b)
    rng = np.random.default_rng(seed)
    diffs = []
    for _ in range(n):
        i = rng.integers(0, len(t), len(t))
        diffs.append((f1_score(t[i], b[i], average='macro', zero_division=0)
                      - f1_score(t[i], a[i], average='macro', zero_division=0)) * 100)
    diffs = np.array(diffs)
    return {'Difference (pts)': diffs.mean(), '95% CI low': np.percentile(diffs, 2.5),
            '95% CI high': np.percentile(diffs, 97.5), 'Share of resamples better': (diffs > 0).mean()}


comparisons = [('Baseline', 'EmoToken'), ('Baseline', 'EmoFusion'), ('EmoToken', 'EmoFusion'),
               ('EmoToken (no <s> in prompt)', 'EmoToken')]
bootstrap = pd.DataFrame([{'Comparison': f'{b} vs {a}', **paired_bootstrap(targets, predicted[a], predicted[b])}
                          for a, b in comparisons]).round(3)
bootstrap

,Comparison,Difference (pts),95% CI low,95% CI high,Share of resamples better
0,EmoToken vs Baseline,9.330,7.720,11.045,1.000
1,EmoFusion vs Baseline,9.818,7.883,11.597,1.000
2,EmoFusion vs EmoToken,0.488,-1.279,2.224,0.703
3,EmoToken vs EmoToken (no <s> in prompt),8.693,7.417,10.187,1.000


### Examples: every model on the same conversation

In [11]:
rng = random.Random(7)
for emo in ['happiness', 'sadness', 'anger', 'surprise', 'fear', 'neutral']:
    idx = [i for i, t in enumerate(targets) if t == emo]
    if not idx:
        continue
    i = rng.choice(idx)
    print(f'TARGET EMOTION: {emo.upper()}')
    print('Context  :', contexts[i][-200:])
    print(f"{'Reference':<28}: {references[i]}  [{predicted['Human reference'][i]}]")
    for row in generated:
        print(f'{row:<28}: {generated[row][i]}  [{predicted[row][i]}]')
    print('-' * 110)

TARGET EMOTION: HAPPINESS
Context  : Hello , Mrs.White ! I'm Anne . [SEP] Nice to see you with us , Anne.Please make yourself at home .
Reference                   : Thank you.You ' re so considerate .  [happiness]
Baseline                    : Thank you very much .  [happiness]
EmoToken                    : Thank you very much .  [happiness]
EmoToken (no <s> in prompt) : Thank you very much .  [happiness]
EmoFusion                   : Thank you very much .  [happiness]
--------------------------------------------------------------------------------------------------------------
TARGET EMOTION: SADNESS
Context  : Hello ! [SEP] Hello , Is that Stefan ?
Reference                   : I'm sorry , but I'm afraid you've dialed the wrong number.There is no Stefan here .  [fear]
Baseline                    : Yes , it is .  [neutral]
EmoToken                    : Yes , it is .  [neutral]
EmoToken (no <s> in prompt) : Yes , it is .  [neutral]
EmoFusion                   : Yes , it's me .  [neutr

## 10. Save the results

In [12]:
results.to_csv(OUT_DIR / 'results.csv')
bootstrap.to_csv(OUT_DIR / 'bootstrap.csv', index=False)
all_out = pd.DataFrame({'id': [p['id'] for p in test_pairs], 'context': contexts,
                        'reference': references, 'target_emotion': targets})
for row in generated:
    all_out[row] = generated[row]
    all_out[row + ' [classified]'] = predicted[row]
all_out.to_csv(OUT_DIR / 'all_test_outputs.csv', index=False)

summary = (f"## Results ({len(test_pairs)} test examples)\n\n{results.to_markdown()}\n\n"
           f"## Paired bootstrap, 7-class macro-F1\n\n{bootstrap.to_markdown(index=False)}\n")
(OUT_DIR / 'results.md').write_text(summary)
print(summary)
print('Saved to', OUT_DIR)

## Results (6740 test examples)

| Model                       |   BLEU-4 |   ROUGE-L |   METEOR |   Distinct-1 |   Distinct-2 |   Sentiment coherence |   Emotion macro-F1 (7) |   Emotion acc (7) |   Emotion macro-F1 (3) |   Emotion acc (3) |
|:----------------------------|---------:|----------:|---------:|-------------:|-------------:|----------------------:|-----------------------:|------------------:|-----------------------:|------------------:|
| Baseline                    |     2.73 |     15.89 |    15.39 |         4.08 |        18.66 |                 53.12 |                  18.6  |             54.69 |                  38.66 |             56.34 |
| EmoToken                    |     3.05 |     16.57 |    16.2  |         4.09 |        19.11 |                 51.74 |                  27.95 |             63.18 |                  47.54 |             64.38 |
| EmoToken (no <s> in prompt) |     2.77 |     15.85 |    15.37 |         4.1  |        18.89 |                 53.01 |        